In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#############################################
# 공통: AttentionLayer 정의 (이름 그대로 두어도 ok)
#############################################
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        score   = self.V(torch.tanh(self.W(x)))    # [B, L, 1]
        weights = F.softmax(score, dim=1)          # [B, L, 1]
        return (weights * x).sum(dim=1)            # [B, hidden_dim]

#############################################
# 텍스트 모델: CNN → BiLSTM → Attention
#############################################
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_filters, kernel_size, lstm_units, dropout_rate, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 이름을 체크포인트 키에 맞춤
        self.conv    = nn.Conv1d(embedding_dim, num_filters, kernel_size, padding=kernel_size//2)
        self.bn      = nn.BatchNorm1d(num_filters)
        self.bilstm  = nn.LSTM(num_filters, lstm_units, batch_first=True, bidirectional=True)
        self.attn    = AttentionLayer(hidden_dim=2*lstm_units)
        
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1     = nn.Linear(2*lstm_units, 64)
        self.fc2     = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x: [B, L]
        x = self.embedding(x)             # [B, L, E]
        x = x.permute(0,2,1)              # [B, E, L]
        x = self.bn(self.conv(x))         # [B, F, L]
        x = x.permute(0,2,1)              # [B, L, F]
        x, _ = self.bilstm(x)             # [B, L, 2U]
        x     = self.attn(x)              # [B, 2U]
        x     = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)                # [B, C]

#############################################
# 이미지 모델 (변경 없음)
#############################################
class ImageEmotionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 32, 3, padding=1)
        self.pool    = nn.MaxPool2d(2,2)
        self.conv2   = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3   = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1     = nn.Linear(128*28*28, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(128, num_classes)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.conv3(x)); x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x)); x = self.dropout(x)
        return self.fc2(x)

#############################################
# 통합 모델 정의
#############################################
class IntegratedEmotionModel(nn.Module):
    def __init__(self, text_model, image_model, text_weight=2.0, image_weight=1.0):
        super().__init__()
        self.text_model  = text_model
        self.image_model = image_model
        self.tw, self.iw = text_weight, image_weight
    
    def forward(self, text_input, image_input):
        t_logits = self.text_model(text_input)
        i_logits = self.image_model(image_input)
        t_probs  = F.softmax(t_logits, dim=1) * self.tw
        i_probs  = F.softmax(i_logits, dim=1) * self.iw
        return (t_probs + i_probs) / (self.tw + self.iw)

#############################################
# ★ 하이퍼파라미터 (학습 때와 동일하게)
#############################################
vocab_size     = 50000
embedding_dim  = 300
num_filters    = 64
kernel_size    = 3
lstm_units     = 128
dropout_rate   = 0.3
num_classes    = 7
max_seq_length = 98

# 1) 텍스트 모델 로드
text_model = CNN_BiLSTM_Attention(
    vocab_size, embedding_dim,
    num_filters, kernel_size,
    lstm_units, dropout_rate,
    num_classes
)
text_model.load_state_dict(torch.load("saved_models/best_text_model.pt", map_location="cpu"))
text_model.eval()

# 2) 이미지 모델 로드
image_model = ImageEmotionModel(num_classes)
image_model.load_state_dict(torch.load("emotion_image_model.pt", map_location="cpu"))
image_model.eval()

# 3) 통합 모델
integrated = IntegratedEmotionModel(text_model, image_model, text_weight=2.0, image_weight=1.0)
integrated.eval()

# 4) TorchScript 변환 및 저장
example_text  = torch.randint(0, vocab_size, (1, max_seq_length))
example_image = torch.randn(1, 3, 224, 224)
scripted = torch.jit.trace(integrated, (example_text, example_image))
torch.jit.save(scripted, "integrated_emotion_model.pt")
print("✅ Saved TorchScript model to integrated_emotion_model.pt")

